# 11. Dashboard evaluation

| | |
|---|---|
| **Stage** | 11 of 11 |
| **Purpose** | Evaluate the published dashboard at two levels. Offline: recompute every number it shows from its own base tables. Online: open every page and the main filters automatically, check that the page shows the recomputed top issue, and request the live link without logging in. |
| **Inputs** | `the public dashboard repository (cloned from GitHub): data/, src/aspect_taxonomy.py, app.py`<br>`the live dashboard URL` |
| **Outputs** | `dashboard_eval_offline.csv`<br>`dashboard_eval_functional.csv`<br>`dashboard_eval_live.csv` (in `/content/dashboard_evaluation`, or on your own Google Drive if `SAVE_TO_DRIVE = True`) |
| **Runtime** | CPU only (about 2 minutes). |
| **Depends on** | None (it reads the public GitHub repository) |
| **Configuration** | No Google Drive is needed: the notebook reads the public repository and the public dashboard link. Set `SAVE_TO_DRIVE = True` in the first code cell only if you want to keep the result tables on your own Drive. |

## What the cell checks

**A. Offline correctness (25 checks).** The totals of reviews, sentences and pairs; the uniqueness and ranges of the pair table; the issue-priority table recomputed from the pairs and sentence counts (negative mentions and mentions per 100 sentences, the top issue of every game, the cross-game table); the trend table summed over all windows against the pair counts (all markets and US only); the example sentences against the pair table; the version windows recomputed from the review table; and a privacy scan of the dashboard text.

**B. Functional test (37 scenarios).** All eight pages, five games on the three game-specific pages, and the main filters are opened with Streamlit's test runner, which simulates user actions without a browser. The cell also asserts that the totals on the Overview page match the documented totals and that the top issue shown for each game has the highest recomputed priority. Rendering times are measured on the machine that runs the notebook, not on Streamlit Cloud.

**C. Live link.** Two plain HTTP requests to the public URL, without logging in: the status code and the response time.

**Limits.** The test runner does not check visual layout or browser rendering. A user task test (new users answering fixed questions) is separate and manual. The Streamlit Community Cloud health path returns the app page, so it is not a separate health signal.

In [ ]:
# ===== Dashboard evaluation: (A) offline correctness checks, (B) functional test of every page, (C) live-link check =====
# Reads the PUBLIC repository (nothing is modified); no Google Drive is needed. Writes three small CSV files to OUT_DIR.
%pip install -q streamlit plotly
import os, re, sys, time, shutil, subprocess, importlib
import numpy as np, pandas as pd

REPO_URL = "https://github.com/Kun0808/gamesense-dashboard.git"
LIVE_URL = "https://gamesense-dashboard-plp.streamlit.app"
WORK = "/content/dashboard_eval_repo"
OUT_DIR = "/content/dashboard_evaluation"                # result tables are written here (temporary Colab storage)
SAVE_TO_DRIVE = False                                    # True = write them to your own Google Drive instead
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/GameSense/outputs/dashboard_evaluation"
EXPECT = {"reviews": 33977, "sentences": 62466, "pairs": 34800}          # the documented totals
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------- 0. fresh copy of the repository
shutil.rmtree(WORK, ignore_errors=True)
if REPO_URL.startswith("http"):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], capture_output=True, text=True)
    assert r.returncode == 0, "git clone failed (is the repository public?): " + r.stderr[-300:]
else:
    shutil.copytree(REPO_URL, WORK)
D = f"{WORK}/data"
meta, pairs, ex, trend = (pd.read_parquet(f"{D}/reviews_meta.parquet"), pd.read_parquet(f"{D}/pairs.parquet"), pd.read_parquet(f"{D}/examples.parquet"), pd.read_csv(f"{D}/trend.csv"))
prio = pd.read_csv(f"{D}/issue_priority_by_game.csv"); cross = pd.read_csv(f"{D}/cross_game_common_priority.csv", index_col=0)
vers = pd.read_csv(f"{D}/version_windows_and_stars.csv", dtype={"version": str})
sys.path.insert(0, f"{WORK}/src"); import aspect_taxonomy as T
GAMES = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]; CAUTION = {"advertisement", "account_support"}
COMMON = ["performance_stability", "monetization_value", "advertisement", "level_difficulty_balance", "content_progression", "paywall", "account_support"]

# ---------------------------------------------------------------- A. offline correctness: recompute what the dashboard shows from its own base tables
rows = []
def check(area, name, ok, detail=""):
    rows.append({"area": area, "check": name, "result": "PASS" if ok else "FAIL", "detail": detail})
meta["at"] = pd.to_datetime(meta["at"], errors="coerce")
check("data", "reviews: unique ids and documented total", meta["reviewId"].is_unique and len(meta) == EXPECT["reviews"], f"{len(meta):,} reviews")
check("data", "sentences: sum of n_sentences and documented total", int(meta["n_sentences"].sum()) == EXPECT["sentences"], f"{int(meta['n_sentences'].sum()):,} sentences")
check("data", "pairs: documented total", len(pairs) == EXPECT["pairs"], f"{len(pairs):,} pairs")
check("data", "no duplicate (sentence, aspect) pair", not pairs.duplicated(["sentence_uid", "aspect"]).any())
check("data", "every pair belongs to a known review", pairs["reviewId"].isin(meta["reviewId"]).all())
check("data", "sentiment labels and probability range", set(pairs["sentiment"].unique()) <= {"negative", "neutral", "positive"} and pairs["p_negative"].between(0, 1).all())
check("data", "star ratings between 1 and 5", pairs["stars"].between(1, 5).all() and meta["score"].between(1, 5).all())
check("data", "all aspects are in the taxonomy", set(pairs["aspect"].unique()) <= set(T.ASPECT_IDS))

S = meta.groupby("game")["n_sentences"].sum()
neg = pairs[pairs["sentiment"] == "negative"].groupby(["game", "aspect"]).size().rename("neg")
allm = pairs.groupby(["game", "aspect"]).size().rename("mentions")
re_ = pd.concat([allm, neg], axis=1).fillna(0).reset_index(); re_["priority"] = 100 * re_["neg"] / re_["game"].map(S); re_["mentions100"] = 100 * re_["mentions"] / re_["game"].map(S)
mg = prio.merge(re_, on=["game", "aspect"], how="outer", indicator=True)
d1 = (mg["priority_neg_per100"] - mg["priority"]).abs().max(); d2 = (mg["mentions_per100"] - mg["mentions100"]).abs().max()
check("priority", "every (game, aspect) of the ranking is recomputable", (mg["_merge"] == "both").all(), f"{(mg['_merge'] != 'both').sum()} unmatched")
check("priority", "negative mentions per 100 sentences equals the recomputed value", d1 < 0.01, f"max abs difference {d1:.4f}")
check("priority", "mentions per 100 sentences equals the recomputed value", d2 < 0.01, f"max abs difference {d2:.4f}")
top_re = re_.sort_values("priority", ascending=False).groupby("game").head(1).set_index("game")["aspect"].to_dict()
top_tb = prio.sort_values("priority_neg_per100", ascending=False).groupby("game").head(1).set_index("game")["aspect"].to_dict()
mx = re_.groupby("game")["priority"].max()
ok_top = all(abs(re_[(re_.game == g) & (re_.aspect == top_tb[g])]["priority"].iloc[0] - mx[g]) < 1e-6 for g in GAMES)
check("priority", "the top issue of every game has the highest recomputed priority", ok_top, "; ".join(f"{g}: {top_tb[g]}" for g in GAMES))
cg = cross.reindex(COMMON)
dd = max(abs(cg.loc[a, g] - prio[(prio.game == g) & (prio.aspect == a)]["priority_neg_per100"].iloc[0]) for a in COMMON for g in GAMES if g in cg.columns and not prio[(prio.game == g) & (prio.aspect == a)].empty and pd.notna(cg.loc[a, g]))
check("priority", "cross-game table equals the per-game ranking for the shared aspects", dd < 0.01, f"max abs difference {dd:.4f}")

for scope, sel in (("all", pairs), ("us", pairs[pairs["market"] == "us"])):
    t = trend[trend["market_scope"] == scope]
    for col, df in (("negative_mentions", sel[sel["sentiment"] == "negative"]), ("mentions", sel)):
        a = t.groupby(["game", "aspect"])[col].sum(); b = df.groupby(["game", "aspect"]).size()
        b = b[b.index.isin(a.index)]; ok = (a.reindex(b.index).fillna(-1) == b).all()
        check("trend", f"{scope}: sum of {col} over all windows equals the pair count", bool(ok), f"{len(b)} (game, aspect) combinations")
check("trend", "windows with fewer than 100 reviews are flagged", ((trend["n_reviews"] < 100) == (~trend["reliable_window"].astype(bool))).all())

nk = pairs.loc[pairs["sentiment"] == "negative", ["game", "aspect", "sentence_text"]].drop_duplicates()
m = ex.merge(nk, on=["game", "aspect", "sentence_text"], how="left", indicator=True)
check("examples", "every example is a negative pair of the same game and aspect", (m["_merge"] == "both").all(), f"{(m['_merge'] != 'both').sum()} of {len(ex)} not found")
check("examples", "flagged aspects only show 1-2 star reviews", ex[ex["aspect"].isin(CAUTION)]["stars"].le(2).all())

for _, r in vers.iterrows():
    d = meta[meta["game"] == r["game"]].dropna(subset=["at", "score"]); on = d[d["version"].astype(str) == str(r["version"])]
    t0 = on["at"].quantile(0.05); w = pd.Timedelta(days=14)
    pre, post = d[(d["at"] >= t0 - w) & (d["at"] < t0)], d[(d["at"] >= t0) & (d["at"] < t0 + w)]
    ok = len(pre) == r["n_pre"] and len(post) == r["n_post"] and abs(pre["score"].mean() - r["stars_pre"]) < 0.006 and abs(post["score"].mean() - r["stars_post"]) < 0.006
    check("version", f"{r['game']} {r['version']}: window sizes and mean stars recomputed", bool(ok), f"pre {len(pre)} vs {r['n_pre']}, post {len(post)} vs {r['n_post']}, stars {pre['score'].mean():.2f}/{post['score'].mean():.2f} vs {r['stars_pre']}/{r['stars_post']}")

PII = [re.compile(p) for p in (r"[\w.+-]+@[\w-]+\.[\w.-]+", r"https?://\S+|www\.\S+", r"(?<![\w@])@\w{3,}", r"(?<!\d)\d{8,}(?!\d)", r"(?<!\d)\+?\d[\d\s().-]{7,}\d(?!\d)")]
left = sum(int(df[c].astype("string").str.contains(p).fillna(False).sum()) for df in (pairs, ex) for c in ("sentence_text", "content_clean") if c in df for p in PII)
check("privacy", "no e-mail, link, @handle or long number in the dashboard text", left == 0, f"{left} matches")
check("privacy", "no user-name column in any dashboard table", not any({"userName", "userImage"} & set(df.columns) for df in (meta, pairs, ex)))
offline = pd.DataFrame(rows); offline.to_csv(f"{OUT_DIR}/dashboard_eval_offline.csv", index=False)
print("=== A. OFFLINE CORRECTNESS ===\n", offline.to_string(index=False)); print(f"\n{(offline['result'] == 'PASS').sum()} of {len(offline)} checks passed")

# ---------------------------------------------------------------- B. functional test: every page and the main filters, with timing
from streamlit.testing.v1 import AppTest
APP = f"{WORK}/app.py"; fr = []
def run(label, at, action=None):
    t0 = time.perf_counter(); err = ""
    try:
        if action: action(at)
        at.run()
        err = "; ".join(str(e.value)[:80] for e in at.exception)
    except Exception as e:
        err = f"{type(e).__name__}: {str(e)[:80]}"
    fr.append({"scenario": label, "seconds": round(time.perf_counter() - t0, 2), "result": "PASS" if not err else "FAIL", "error": err})
def fresh(page=None):
    a = AppTest.from_file(APP, default_timeout=180).run()
    if page: a.sidebar.radio[0].set_value(page).run()
    return a
PAGES = ["Overview", "Issue priority", "Aspect sentiment", "Trends", "Cross-game comparison", "Review explorer", "Version analysis", "Method & reliability"]
for p in PAGES:
    a = AppTest.from_file(APP, default_timeout=180); t0 = time.perf_counter(); a.run(); a.sidebar.radio[0].set_value(p).run()
    fr.append({"scenario": f"open page: {p}", "seconds": round(time.perf_counter() - t0, 2), "result": "PASS" if not a.exception else "FAIL", "error": "; ".join(str(e.value)[:80] for e in a.exception)})
for g in GAMES:
    for page in ("Issue priority", "Aspect sentiment", "Trends"):
        a = fresh(page); run(f"{page}: game = {g}", a, lambda x, g=g: x.main.selectbox[0].set_value(g))
a = fresh("Issue priority"); run("Issue priority: 1-2 star view", a, lambda x: x.main.radio[0].set_value("1-2 star reviews only"))
a = fresh("Trends"); run("Trends: all markets (non-US shading)", a, lambda x: x.main.radio[0].set_value("all"))
a = fresh("Trends"); run("Trends: show update markers", a, lambda x: x.main.checkbox[0].set_value(True))
a = fresh("Aspect sentiment"); run("Aspect sentiment: US only", a, lambda x: x.main.radio[0].set_value("US only"))
a = fresh("Review explorer"); run("Explorer: browse all sentences", a, lambda x: x.main.radio[0].set_value("Browse all matching sentences"))
a = fresh("Review explorer"); run("Explorer: text search", a, lambda x: x.main.text_input[0].set_value("lag"))
a = fresh("Review explorer"); run("Explorer: flagged aspect shows its warning", a, lambda x: x.main.selectbox[1].set_value("advertisement"))
a = fresh("Overview"); run("Overview: US only", a, lambda x: x.main.radio[0].set_value("US only"))
# content assertions on the live data: what the page shows equals the recomputed values
a = fresh("Overview"); vals = {mt.label: mt.value for mt in a.metric}
fr.append({"scenario": "Overview shows the documented review and sentence totals", "seconds": 0.0, "result": "PASS" if vals.get("Reviews") == f"{EXPECT['reviews']:,}" and vals.get("Review sentences") == f"{EXPECT['sentences']:,}" else "FAIL", "error": str(vals)[:120]})
label2id = {T.ASPECT_TAXONOMY[x]["label"].strip().lower(): x for x in T.ASPECT_IDS}
for g in GAMES:
    a = fresh("Issue priority"); a.main.selectbox[0].set_value(g).run()
    shown = next((mt.value for mt in a.metric if mt.label == "Top issue"), ""); sid = label2id.get(shown.strip().lower())
    val = re_[(re_.game == g) & (re_.aspect == sid)]["priority"]
    ok = len(val) > 0 and abs(val.iloc[0] - mx[g]) < 1e-6
    fr.append({"scenario": f"top issue shown for {g} has the highest recomputed priority", "seconds": 0.0, "result": "PASS" if ok else "FAIL",
               "error": f"shown '{shown}' (priority {val.iloc[0]:.2f}), maximum {mx[g]:.2f}" if len(val) else f"shown '{shown}' is not a known aspect"})
func = pd.DataFrame(fr); func.to_csv(f"{OUT_DIR}/dashboard_eval_functional.csv", index=False)
print("\n=== B. FUNCTIONAL TEST (rendering time is measured on this Colab machine, not on Streamlit Cloud) ===\n", func.to_string(index=False))
t = func[func["seconds"] > 0]["seconds"]
print(f"\n{(func['result'] == 'PASS').sum()} of {len(func)} scenarios passed | render time per scenario: median {t.median():.1f} s, max {t.max():.1f} s")

# ---------------------------------------------------------------- C. the live link
live = []
try:
    import requests
    for path in ("", "/_stcore/health"):
        t0 = time.perf_counter(); r = requests.get(LIVE_URL + path, timeout=60)
        live.append({"request": LIVE_URL + path, "status": r.status_code, "seconds": round(time.perf_counter() - t0, 2), "body_starts_with": r.text[:20].replace("\n", " ")})
except Exception as e:
    live.append({"request": LIVE_URL, "status": f"error: {type(e).__name__}", "seconds": None, "body_starts_with": ""})
lv = pd.DataFrame(live); lv.to_csv(f"{OUT_DIR}/dashboard_eval_live.csv", index=False)
print("\n=== C. LIVE LINK (no login, plain request) ===\n", lv.to_string(index=False))
print("\nSaved to", OUT_DIR)


## How to read the output

- **A:** every line should be `PASS`. A `max abs difference` of about 0.0005 is rounding, because the tables store three decimals. A `FAIL` names the check and how far the values differ; first make sure the data files in the repository are the latest ones.
- **B:** every scenario should be `PASS`. The first page open is slower because the data is loaded; later interactions are served from the cache. The `missing ScriptRunContext` warnings come from running Streamlit's test runner outside Streamlit and can be ignored.
- **C:** status 200 on both requests means the public link answers without a login.